Process sessions
Jira: https://keyless.atlassian.net/browse/BIOM-625

In [1]:
import shutil

import numpy as np
import pandas as pd
from loguru import logger

from distutils.dir_util import copy_tree
from pathlib import Path

import cv2
import datasets
import os

import modules.globals
from modules import core
from modules.face_analyser import get_one_face

from tqdm.auto import tqdm

ImportError: libGL.so.1: cannot open shared object file: No such file or directory

In [ ]:
dataset_path = Path(
    "/home/sagemaker-user/hf_datasets/sagemaker-production-eu-central-1-kl-biometric-experiments/pipeline_data/main/lr_dataset_2.2/20250623-113349/filter_sessions/lr_2_7_8_sensors_raw_test/"
)
output_imgs_path = Path("/home/sagemaker-user/face_swap/original_images/")
output_swapped_imgs_path = Path("/home/sagemaker-user/face_swap/swapped_images_enh/")
execution_provider = "cuda"  # cuda or cpu
face_enhancer = True
face_det_th = 0.85
rng_seed = 42

In [ ]:
os.makedirs(output_imgs_path, exist_ok=True)
os.makedirs(output_swapped_imgs_path, exist_ok=True)

In [ ]:
### init ###
modules.globals.execution_providers = core.decode_execution_providers(
    [execution_provider]
)
frame_processors = ["face_swapper"]
modules.globals.fp_ui["face_enhancer"] = False
if face_enhancer:
    frame_processors.append("face_enhancer")
    modules.globals.fp_ui["face_enhancer"] = True
np.random.seed(rng_seed)

modules.globals.max_memory = core.suggest_max_memory()
modules.globals.execution_threads = core.suggest_execution_threads()
core.limit_resources()

In [ ]:
target = "/home/sagemaker-user/face_swap/original_images/1709198987583"
output_dir = "/home/sagemaker-user/face_swap/swapped_images_enh/1709198987583"

In [ ]:
source_frame = "/home/sagemaker-user/face_swap/original_images/T-02048_1650389859936_1.2.0/T26260_T-02048-1650389859936-1.2.0_1.jpg"

In [ ]:
temp_frame_paths = [str(p) for p in Path(output_dir).glob("*.jpg")]


In [ ]:
shutil.copyfile(source_frame, Path(output_dir) / "source_img.jpg")


In [ ]:
for frame_processor in core.get_frame_processors_modules(frame_processors):
    logger.info(f"Progressing... {frame_processor.NAME}")
    print(f"Total frames: {len(temp_frame_paths)}")
    frame_processor.process_video(str(source_frame), temp_frame_paths)
    core.release_resources()

In [ ]:
### process ###
output_swapped_imgs_path.mkdir(parents=True, exist_ok=True)

for _, row in mapping_df.iterrows():
    session_name = row["session_folder_target"]
    target = output_imgs_path / session_name
    output_dir = output_swapped_imgs_path / session_name
    source_frame = row["source_path"]
    output_dir.mkdir(exist_ok=True)
    print(target, output_dir)
    copy_tree(target, output_dir)
    print("source_frame", source_frame)
    temp_frame_paths = [str(p) for p in output_dir.glob("*.jpg")]
    print(temp_frame_paths[0])
    shutil.copyfile(source_frame, output_dir / "source_img.jpg")
    for frame_processor in core.get_frame_processors_modules(frame_processors):
        logger.info(f"Progressing... {frame_processor.NAME}")
        print(f"Total frames: {len(temp_frame_paths)}")
        frame_processor.process_video(str(source_frame), temp_frame_paths)
        core.release_resources()